# ImageNet: A Large-Scale Hierarchical Image Database (Deng et al., CVPR 2009)
### Triển Khai Phân Loại Hình Ảnh Phân Cấp Bằng PyTorch & Đối Chiếu Giá Trị Học Thuật

> **Tác giả:** Jia Deng, Wei Dong, Richard Socher, Li-Jia Li, Kai Li, Li Fei-Fei  
> **Hội nghị:** CVPR 2009 (Miami, Florida)  
> **Ý nghĩa:** Chuyển đổi trọng tâm từ Thuật toán sang Dữ liệu lớn (Data-Centric AI), xác lập chuẩn đánh giá Top-1 / Top-5 và Loss phân cấp.

---

## 🎯 Mục Tiêu Bài Thực Hành
1. **Hierarchical Cross-Entropy Loss:** Cài đặt hàm mất mát phạt lỗi theo khoảng cách nhảy cây phả hệ WordNet.
2. **Vision CNN:** Xây dựng mạng nơ-ron tích chập thị giác tiếp nhận ảnh RGB và trích xuất biểu diễn ngữ nghĩa.
3. **Chuẩn đánh giá ImageNet:** Tính toán Top-1 Accuracy, Top-5 Accuracy và Mean Tree-Hop Distance.
4. **Academic Value Benchmark:** Thực thi đối chiếu 3 mô hình đại diện cho 3 giai đoạn:
   - *Mô hình 1 (Kỷ nguyên Tiền-ImageNet):* Đặc trưng thủ công (Color Histograms / SIFT-like) + Bộ phân loại tuyến tính.
   - *Mô hình 2 (CNN Dữ liệu nhỏ - Caltech-101 scale):* Quá khớp (Overfitting) trầm trọng do thiếu dữ liệu đa dạng.
   - *Mô hình 3 (Tư duy ImageNet Scale + Tree Loss):* Học biểu diễn sâu phong phú, thu hẹp khoảng cách tổng quát hóa.
5. **Trực quan hóa:** Xuất biểu đồ phân tích 4 bảng đối chiếu chuyên sâu.

In [ ]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[✓] Thiết bị tính toán được cấu hình: {device}")

## PHẦN 1: HÀM MẤT MÁT PHÂN CẤP (HIERARCHICAL CROSS-ENTROPY LOSS)

Cross-Entropy truyền thống xem mọi sai số là như nhau:
$$\mathcal{L}_{CE} = -\log P(y_{\text{true}})$$

Hàm mất mát phân cấp của ImageNet bổ sung thành phần phạt khoảng cách cây phả hệ WordNet:
$$\mathcal{L}_{\text{total}} = \mathcal{L}_{CE} + \lambda \sum_{j=1}^C P(y = j) \cdot D_{\text{tree}}(y_{\text{true}}, j)$$

- Nếu nhầm lẫn giữa hai giống chó cùng nhánh: $D_{\text{tree}} = 2$ $\rightarrow$ Phạt nhẹ.
- Nếu nhầm lẫn giữa con chó và chiếc ô tô: $D_{\text{tree}} = 8$ $\rightarrow$ Phạt nặng!

In [ ]:
class HierarchicalCrossEntropyLoss(nn.Module):
    """
    Hàm mất mát phân cấp phạt sai số theo khoảng cách nhảy cây WordNet.
    """
    def __init__(self, distance_matrix, penalty_weight=0.15):
        super(HierarchicalCrossEntropyLoss, self).__init__()
        if not isinstance(distance_matrix, torch.Tensor):
            self.register_buffer("distance_matrix", torch.tensor(distance_matrix, dtype=torch.float32))
        else:
            self.register_buffer("distance_matrix", distance_matrix.float())
        self.penalty_weight = penalty_weight
        self.ce_loss = nn.CrossEntropyLoss()

    def forward(self, logits, targets):
        base_ce = self.ce_loss(logits, targets)
        probs = torch.softmax(logits, dim=1)
        target_dists = self.distance_matrix[targets]
        hierarchical_penalty = torch.mean(torch.sum(probs * target_dists, dim=1))
        total_loss = base_ce + self.penalty_weight * hierarchical_penalty
        return total_loss, base_ce, hierarchical_penalty

print("[✓] Đã khởi tạo HierarchicalCrossEntropyLoss thành công.")

## PHẦN 2: KIẾN TRÚC MÔ HÌNH THỊ GIÁC

1. **HierarchicalVisionCNN:** Mạng nơ-ron tích chập đa tầng (Conv-BatchNorm-ReLU-MaxPool) có khả năng học biểu diễn từ thấp đến cao (cạnh $\rightarrow$ họa tiết $\rightarrow$ bộ phận $\rightarrow$ vật thể).
2. **HandcraftedFeatureClassifier:** Trích xuất lược đồ màu toàn cục (Color Histogram) kết hợp phân loại tuyến tính (đại diện cho kỹ thuật Tiền-ImageNet trước năm 2009).

In [ ]:
class HierarchicalVisionCNN(nn.Module):
    def __init__(self, num_classes=8, in_channels=3):
        super(HierarchicalVisionCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        feat = self.features(x)
        return self.classifier(feat)


class HandcraftedFeatureClassifier(nn.Module):
    """Mô phỏng trường phái Tiền-ImageNet: Trích xuất lược đồ màu thủ công + Tuyến tính."""
    def __init__(self, num_classes=8):
        super(HandcraftedFeatureClassifier, self).__init__()
        self.classifier = nn.Sequential(
            nn.Linear(24, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_classes)
        )

    def extract_color_histograms(self, x):
        B, C, H, W = x.shape
        features = []
        for i in range(B):
            sample = x[i].view(C, -1)
            hists = []
            for c in range(C):
                channel_data = sample[c]
                hist = torch.histc(channel_data, bins=8, min=-2.0, max=2.0)
                hists.append(hist / (hist.sum() + 1e-8))
            features.append(torch.cat(hists))
        return torch.stack(features)

    def forward(self, x):
        feats = self.extract_color_histograms(x)
        return self.classifier(feats)

print("[✓] Đã định nghĩa kiến trúc Vision CNN và Handcrafted Feature Extractor.")

## PHẦN 3: BỘ ĐO LƯỜNG CHUẨN MỰC IMAGENET

- **Top-1 Accuracy:** Nhãn có xác suất cao nhất phải đúng.
- **Top-5 Accuracy:** Nhãn thực tế nằm trong 5 dự đoán hàng đầu.
- **Mean Tree-Hop Distance:** Đo khoảng cách trung bình đến nhãn thực tế trên cây WordNet.

In [ ]:
def evaluate_imagenet_metrics(logits, targets, distance_matrix):
    with torch.no_grad():
        probs = torch.softmax(logits, dim=1)
        k = min(5, probs.size(1))

        # Top-1
        _, top1_preds = torch.max(probs, dim=1)
        top1_acc = (top1_preds == targets).float().mean().item()

        # Top-5
        _, topk_preds = torch.topk(probs, k=k, dim=1)
        topk_correct = topk_preds.eq(targets.view(-1, 1).expand_as(topk_preds))
        top5_acc = topk_correct.any(dim=1).float().mean().item()

        # Mean Tree-Hop Distance
        dists = distance_matrix[targets, top1_preds]
        mean_tree_dist = dists.float().mean().item()

    return top1_acc, top5_acc, mean_tree_dist

print("[✓] Đã sẵn sàng hàm evaluate_imagenet_metrics.")

## PHẦN 4: DỮ LIỆU PHÂN CẤP TỔNG HỢP (SYNTHETIC HIERARCHICAL DATA)

In [ ]:
from imagenet_pytorch import create_synthetic_hierarchical_data

X, y, dist_matrix_np, class_names = create_synthetic_hierarchical_data(n_samples=1000, n_classes=8)
print(f"[*] Kích thước tập dữ liệu mẫu: X = {X.shape}, y = {y.shape}")
print(f"[*] Danh sách các lớp phân cấp:")
for i, name in enumerate(class_names):
    branch = 'Living Thing' if i < 4 else 'Artifact'
    print(f"    [{i}] {name:<18} (Nhánh: {branch})")

## PHẦN 5: ĐỐI CHIẾU GIÁ TRỊ HỌC THUẬT (ACADEMIC VALUE BENCHMARK)

Chúng ta sẽ cùng chạy và so sánh 3 mô hình tương ứng với 3 giai đoạn phát triển của ngành Thị giác Máy tính:
1. **Mô Hình 1: Hand-crafted Features (Kỷ nguyên Tiền-ImageNet):** Hạn chế về biểu diễn, không thể mô tả được thế giới thị giác phong phú.
2. **Mô Hình 2: Vision CNN trên tập dữ liệu nhỏ (Caltech-101 scale):** Gặp hiện tượng Overfitting trầm trọng (Train acc cao, Val acc thấp).
3. **Mô Hình 3: Vision CNN trên ImageNet Scale + Hierarchical Loss:** Tận dụng dữ liệu lớn phong phú kết hợp phạt lỗi phân cấp, mang lại hiệu năng tổng quát hóa vượt trội.

In [ ]:
from imagenet_pytorch import run_academic_value_benchmark

# Chạy thử nghiệm đối chiếu học thuật
bench_data = run_academic_value_benchmark(device)

## PHẦN 6: TRỰC QUAN HÓA BIỂU ĐỒ ĐỐI CHIẾU CHUYÊN SÂU

Xuất biểu đồ 4 bảng phân tích:
1. **Top-1 vs Top-5 Accuracy:** Thể hiện ưu thế của việc đánh giá Top-5 trên số lượng lớp lớn.
2. **Khoảng cách Quá Khớp (Generalization Gap):** Minh chứng dữ liệu quy mô lớn giải quyết triệt để điểm nghẽn Overfitting.
3. **Mean Tree-Hop Distance:** Đo lường mức độ văn minh của mô hình trong việc nhầm lẫn ngữ nghĩa.
4. **Ma trận Nhầm lẫn Phân cấp:** Trực quan hóa tỷ lệ nhầm lẫn trong nội bộ nhánh so với nhầm lẫn xuyên miền.

In [ ]:
from imagenet_pytorch import plot_comprehensive_academic_comparison

out_dir = "outputs"
plot_comprehensive_academic_comparison(bench_data, output_dir=out_dir)

# Hiển thị biểu đồ trực tiếp trên notebook
img_path = os.path.join(out_dir, "imagenet_academic_comparison.png")
if os.path.exists(img_path):
    img = plt.imread(img_path)
    plt.figure(figsize=(16, 12))
    plt.imshow(img)
    plt.axis('off')
    plt.title("Kết Quả Đối Chiếu Học Thuật Toàn Diện (ImageNet CVPR 2009)", fontsize=14, fontweight='bold')
    plt.show()

## 🎓 KẾT LUẬN & ĐÓNG GÓP LỊCH SỬ CỦA IMAGENET (CVPR 2009)

1. **Chuyển dịch sang AI Lấy Dữ Liệu Làm Trung Tâm (Data-Centric AI):** ImageNet đã đảo ngược niềm tin cũ rằng "chỉ cần thuật toán thông minh". Dữ liệu lớn, sạch và đa dạng chính là điều kiện tiên quyết để giải quyết hiện tượng quá khớp (overfitting).
2. **Chuẩn Mực Đánh Giá Top-1 & Top-5:** Đưa ra phương pháp đánh giá thực tế và công bằng hơn khi phân loại hàng nghìn danh mục có sự phân nhánh tự nhiên.
3. **Bệ Phóng Của Cách Mạng Deep Learning (AlexNet 2012):** Thử thách ILSVRC dựa trên ImageNet chính là chất xúc tác đưa mạng nơ-ron tích chập sâu và tính toán GPU bước lên đỉnh cao công nghệ toàn cầu.